In [2]:
import polars as pl
from pathlib import Path

file_path = Path("data_cepal.csv")

df = pl.read_csv(file_path)

df.head()

indicator,País__ESTANDAR,Grupos etarios Uso Internet,Años__ESTANDAR,value,unit,notes_ids,source_id
str,str,str,i64,i64,str,str,i64
"""Personas usuarias de Internet …","""Argentina""","""edad de medicion a 17 años""",2016,76,"""Porcentaje sobre el total de p…",null,9353
"""Personas usuarias de Internet …","""Argentina""","""edad de medicion a 17 años""",2017,76,"""Porcentaje sobre el total de p…",null,9353
"""Personas usuarias de Internet …","""Argentina""","""edad de medicion a 17 años""",2018,79,"""Porcentaje sobre el total de p…",null,9353
"""Personas usuarias de Internet …","""Argentina""","""edad de medicion a 17 años""",2019,79,"""Porcentaje sobre el total de p…",null,9353
"""Personas usuarias de Internet …","""Argentina""","""edad de medicion a 17 años""",2020,88,"""Porcentaje sobre el total de p…",null,9353


### Paso 1: Eliminación de Columnas Irrelevantes
Para que un algoritmo de clustering sea efectivo, es crucial realizar una limpieza selectiva de las variables. Las columnas `indicator`, `unit`, `notes_ids` y `source_id` contienen metadatos administrativos o valores descriptivos que son constantes para todos los registros. Al eliminar esta información redundante, se reduce el "ruido" en los datos, permitiendo que el modelo se enfoque exclusivamente en las variables que realmente diferencian el comportamiento de uso de internet entre los grupos.

In [3]:
df = df.drop(["indicator", "unit", "notes_ids", "source_id"])

df.head()

País__ESTANDAR,Grupos etarios Uso Internet,Años__ESTANDAR,value
str,str,i64,i64
"""Argentina""","""edad de medicion a 17 años""",2016,76
"""Argentina""","""edad de medicion a 17 años""",2017,76
"""Argentina""","""edad de medicion a 17 años""",2018,79
"""Argentina""","""edad de medicion a 17 años""",2019,79
"""Argentina""","""edad de medicion a 17 años""",2020,88


### Paso 2: Renombramiento de Columnas
La estandarización de nombres es un paso esencial para la mantenibilidad del código. Se renombran las columnas a un formato técnico y sin caracteres especiales. Esto no solo facilita la escritura del código y evita errores de sintaxis, sino que también asegura que los datos sean compatibles con diversas herramientas de análisis y visualización que prefieren convenciones de nomenclatura uniformes.

In [4]:
df = df.rename({
    "País__ESTANDAR": "country",
    "Grupos etarios Uso Internet": "ageGroup",
    "Años__ESTANDAR": "year",
    "value": "percentage"
})

df.head()

country,ageGroup,year,percentage
str,str,i64,i64
"""Argentina""","""edad de medicion a 17 años""",2016,76
"""Argentina""","""edad de medicion a 17 años""",2017,76
"""Argentina""","""edad de medicion a 17 años""",2018,79
"""Argentina""","""edad de medicion a 17 años""",2019,79
"""Argentina""","""edad de medicion a 17 años""",2020,88


### Paso 3: Pivotaje de la Tabla y Guardado
El paso final y más crítico es la reestructuración de la tabla mediante un pivotaje. Los algoritmos de clustering requieren que cada fila represente una entidad única a comparar y cada columna sea un atributo específico. Al transformar los grupos etarios de filas a columnas, creamos una "matriz de características" donde cada país y año tiene su propio perfil de porcentajes. Esta estructura permite medir matemáticamente la similitud entre observaciones y detectar patrones de comportamiento similares entre distintas regiones y periodos.

In [5]:
df_pivoted = df.pivot(
    index=["country", "year"],
    on="ageGroup",
    values="percentage"
)

output_path = Path("data_per_age_group.csv")
df_pivoted.write_csv(output_path)

df_pivoted.head()

country,year,edad de medicion a 17 años,18 a 25 años de edad,26 a 50 años de edad,51 a 65 años,66 años en adelante,Total
str,i64,i64,i64,i64,i64,i64,i64
"""Argentina""",2016,76,86,82,61,29,71
"""Argentina""",2017,76,90,86,68,34,74
"""Argentina""",2018,79,90,88,74,40,78
"""Argentina""",2019,79,92,90,77,46,80
"""Argentina""",2020,88,95,92,81,55,86
